In [1]:
import os
print(os.listdir())

['.config', 'yellow_tripdata_2023-08.parquet', 'namma_yatri_open_data.csv', 'yellow_tripdata_2023-06.parquet', 'yellow_tripdata_2023-07.parquet', 'Taxi_Trips_-_2023.csv', 'sample_data']


In [2]:
import pandas as pd

In [3]:
namma_df = pd.read_csv('namma_yatri_open_data.csv')
taxi_df = pd.read_csv('Taxi_Trips_-_2023.csv')

print('Namma shape: ', namma_df.shape)
print('Taxi shape: ', taxi_df.shape)

/tmp/ipykernel_227/1460366695.py:1: DtypeWarning: Columns (0,15,16,30,42,43,52,53,69) have mixed types. Specify dtype option on import or set low_memory=False.
  namma_df = pd.read_csv('namma_yatri_open_data.csv')


Namma shape:  (290283, 76)
Taxi shape:  (377006, 23)


In [4]:
taxi_df.columns

Index(['Trip ID', 'Taxi ID', 'Trip Start Timestamp', 'Trip End Timestamp',
       'Trip Seconds', 'Trip Miles', 'Pickup Census Tract',
       'Dropoff Census Tract', 'Pickup Community Area',
       'Dropoff Community Area', 'Fare', 'Tips', 'Tolls', 'Extras',
       'Trip Total', 'Payment Type', 'Company', 'Pickup Centroid Latitude',
       'Pickup Centroid Longitude', 'Pickup Centroid Location',
       'Dropoff Centroid Latitude', 'Dropoff Centroid Longitude',
       'Dropoff Centroid  Location'],
      dtype='object')

In [5]:
# Convert Trip Start Timestamp to datetime
taxi_df['Trip Start Timestamp'] = pd.to_datetime(
    taxi_df['Trip Start Timestamp'],
    errors='coerce'
)

# Drop invalid dates
taxi_df = taxi_df.dropna(subset=['Trip Start Timestamp'])

print("After cleaning Taxi:", taxi_df.shape)

/tmp/ipykernel_227/1652073779.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  taxi_df['Trip Start Timestamp'] = pd.to_datetime(


After cleaning Taxi: (377005, 23)


In [6]:
taxi_df['date'] = taxi_df['Trip Start Timestamp'].dt.floor('D')
taxi_df['hour'] = taxi_df['Trip Start Timestamp'].dt.hour

taxi_df[['date','hour']].head(10)

,date,hour
0,2023-01-01,0
1,2023-01-01,0
2,2023-01-01,0
3,2023-01-01,0
4,2023-01-01,0
5,2023-01-01,0
6,2023-01-01,0
7,2023-01-01,0
8,2023-01-01,0
9,2023-01-01,0


In [7]:
taxi_hourly = taxi_df.groupby(['date', 'hour']).agg(
    completed_rides=('Trip ID', 'count'),
    total_earnings=('Trip Total', 'sum')
).reset_index()

print("Taxi hourly shape:", taxi_hourly.shape)
taxi_hourly.head()

Taxi hourly shape: (684, 4)


,date,hour,completed_rides,total_earnings
0,2023-01-01,0,517,10537.67
1,2023-01-01,1,650,15005.08
2,2023-01-01,2,618,16011.86
3,2023-01-01,3,390,8885.03
4,2023-01-01,4,243,5557.49


In [8]:
# Convert correct date column
namma_df['date'] = pd.to_datetime(namma_df['date_created'], errors='coerce')

# Drop invalid dates
namma_df = namma_df.dropna(subset=['date'])

# Convert hour properly
namma_df['hour'] = pd.to_numeric(namma_df['hour_created'], errors='coerce')

# Drop invalid hours
namma_df = namma_df.dropna(subset=['hour'])

print("Cleaned Namma shape:", namma_df.shape)

Cleaned Namma shape: (22154, 76)


In [9]:
namma_hourly = namma_df.groupby(['date', 'hour']).agg(
    completed_rides=('no_of_completed_rides', 'sum'),
    total_earnings=('total_earning', 'sum')
).reset_index()

print("Namma hourly shape:", namma_hourly.shape)
namma_hourly.head()

Namma hourly shape: (82, 4)


,date,hour,completed_rides,total_earnings
0,2023-06-10,0.0,436.0,85862.0
1,2023-06-10,1.0,213.0,44927.0
2,2023-06-10,2.0,91.0,19936.0
3,2023-06-10,3.0,66.0,12133.0
4,2023-06-10,4.0,77.0,18605.0


In [10]:
print("Namma min:", namma_hourly['date'].min())
print("Namma max:", namma_hourly['date'].max())
print("Unique dates:", namma_hourly['date'].nunique())

Namma min: 2023-06-10 00:00:00
Namma max: 2023-08-01 00:00:00
Unique dates: 4


In [11]:
# Get Namma date range
start = namma_hourly['date'].min()
end = namma_hourly['date'].max()

print("Using date range:", start, "to", end)


Using date range: 2023-06-10 00:00:00 to 2023-08-01 00:00:00


In [12]:
print("Taxi min date:", taxi_hourly['date'].min())
print("Taxi max date:", taxi_hourly['date'].max())

Taxi min date: 2023-01-01 00:00:00
Taxi max date: 2023-01-29 00:00:00


In [13]:
!pip install pyarrow

In [14]:
import pandas as pd

taxi_june = pd.read_parquet("yellow_tripdata_2023-06.parquet")
taxi_july = pd.read_parquet("yellow_tripdata_2023-07.parquet")
taxi_aug  = pd.read_parquet("yellow_tripdata_2023-08.parquet")

In [15]:
print(taxi_june.shape)
print(taxi_july.shape)
print(taxi_aug.shape)

(3307234, 19)
(2907108, 19)
(2824209, 19)


In [16]:
taxi_df = pd.concat([taxi_june, taxi_july, taxi_aug], ignore_index=True)

print("Taxi combined shape:", taxi_df.shape)

Taxi combined shape: (9038551, 19)


In [17]:
taxi_df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee'],
      dtype='object')

In [18]:
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'])

taxi_df['date'] = taxi_df['tpep_pickup_datetime'].dt.floor('D')
taxi_df['hour'] = taxi_df['tpep_pickup_datetime'].dt.hour

In [19]:
taxi_df[['date','hour']].head()

,date,hour
0,2023-06-01,0
1,2023-06-01,0
2,2023-06-01,0
3,2023-06-01,0
4,2023-06-01,0


In [20]:
taxi_hourly = taxi_df.groupby(['date','hour']).agg(
    completed_rides=('tpep_pickup_datetime','count'),
    total_earnings=('total_amount','sum')
).reset_index()

print(taxi_hourly.shape)
taxi_hourly.head()

(2225, 4)


,date,hour,completed_rides,total_earnings
0,2002-12-31,22,1,51.80
1,2002-12-31,23,4,271.39
2,2003-01-01,0,1,25.38
3,2008-12-31,14,1,62.93
4,2008-12-31,23,4,147.21


In [21]:
print("Taxi min:", taxi_hourly['date'].min())
print("Taxi max:", taxi_hourly['date'].max())

Taxi min: 2002-12-31 00:00:00
Taxi max: 2023-10-16 00:00:00


In [22]:
comparison_df = pd.merge(
    namma_hourly,
    taxi_hourly,
    on=['date','hour'],
    how='inner',
    suffixes=('_namma','_taxi')
)

print(comparison_df.shape)
comparison_df.head()

(82, 6)


,date,hour,completed_rides_namma,total_earnings_namma,completed_rides_taxi,total_earnings_taxi
0,2023-06-10,0.0,436.0,85862.0,5710,149256.26
1,2023-06-10,1.0,213.0,44927.0,4685,112927.04
2,2023-06-10,2.0,91.0,19936.0,3675,86665.25
3,2023-06-10,3.0,66.0,12133.0,2290,54519.34
4,2023-06-10,4.0,77.0,18605.0,1117,28073.63


In [23]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 23.3 MB/s eta 0:00:00


In [24]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="yourpassword",
    database="ride_analysis"
)

cursor = conn.cursor()

InterfaceError: 2003: Can't connect to MySQL server on 'localhost:3306' (Errno 111: Connection refused)

In [25]:
comparison_df.to_csv("ride_comparison.csv", index=False)